In [2]:
import pandas as pd
import psycopg2  # Use sqlite3 for SQLite
import os

# Create folder for results

In [ ]:
os.makedirs("sql_results", exist_ok=True)

# -------------------------------
# Connect to PostgreSQL
# -------------------------------
conn = psycopg2.connect(
    host="localhost",
    dbname="myDataBase",
    user="postgres",
    password="#Your Password"
)

# Define all queries

In [7]:
queries = {
    "top_10_customers": """
        SELECT customer_name, SUM(sales) AS total_sales
        FROM orders
        GROUP BY customer_name
        ORDER BY total_sales DESC
        LIMIT 10;
    """,
    
    "profit_margin_by_category": """
        SELECT category,
               AVG(profit / NULLIF(sales, 0)) AS avg_profit_margin
        FROM orders
        WHERE sales IS NOT NULL
        GROUP BY category
        ORDER BY avg_profit_margin ASC;
    """,
    
    "profit_margin_by_region": """
        SELECT region,
               AVG(profit / NULLIF(sales, 0)) AS avg_profit_margin
        FROM orders
        GROUP BY region
        ORDER BY avg_profit_margin ASC;
    """,
    
    "count_negative_orders": """
        SELECT COUNT(*) AS negative_orders
        FROM orders
        WHERE profit < 0;
    """,
    
    "monthly_sales_trend": """
        SELECT TO_CHAR(order_date, 'YYYY-MM') AS month,
               SUM(sales) AS total_sales
        FROM orders
        GROUP BY TO_CHAR(order_date, 'YYYY-MM')
        ORDER BY month;
    """,
    
    "region_category_profit": """
        SELECT region, category,
               AVG(profit / NULLIF(sales, 0)) AS avg_profit_margin
        FROM orders
        GROUP BY region, category
        ORDER BY avg_profit_margin ASC;
    """,
    
    "top_customers_ranking": """
        SELECT customer_name,
               SUM(sales) AS total_sales,
               RANK() OVER (ORDER BY SUM(sales) DESC) AS rank
        FROM orders
        GROUP BY customer_name;
    """,
    
    "profit_by_discount_level": """
        SELECT 
            CASE 
                WHEN discount = 0 THEN 'No Discount'
                WHEN discount <= 0.2 THEN 'Low Discount'
                ELSE 'High Discount'
            END AS discount_category,
            AVG(profit / NULLIF(sales, 0)) AS avg_profit_margin
        FROM orders
        GROUP BY discount_category;
    """
}

# Run all queries and save CSVs

In [ ]:
for name, query in queries.items():
    df = pd.read_sql(query, conn)
    filename = f"sql_results/{name}.csv"
    df.to_csv(filename, index=False)
    print(f"Saved {filename} ({len(df)} rows)")

# Close connection

In [9]:
conn.close()
print("All queries executed and results saved!")

All queries executed and results saved!
